In [2]:
# --- Data Loading and Preprocessing ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# You will need to upload 'space_debris_ml_ready.csv' to your Colab session.
# Assuming the new dataset is named 'space_debris_new.csv' or similar,
# please update the filename below if necessary.
file_path = 'space_debris_ml_ready.csv' # <--- Update this if your new file has a different name

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    print("Original columns:", df.columns.tolist())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    print("Please ensure the file is uploaded to your Colab session.")
    exit()

# Engineering features from the original dataset
# Ensure that the necessary columns for feature engineering exist in the new dataset
required_cols = ['MEAN_MOTION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'NORAD_CAT_ID', 'EPOCH_UNIX', 'ECCENTRICITY', 'INCLINATION']
if not all(col in df.columns for col in required_cols):
    missing = [col for col in required_cols if col not in df.columns]
    print(f"Error: The dataset is missing required columns for feature engineering: {missing}")
    # Attempt to proceed with available columns, but be aware of potential issues
    # exit() # Uncomment this line to stop execution if critical columns are missing

R_EARTH = 6371.0
MU = 398600.4418

# Re-calculate engineered features to ensure they are present
if 'MEAN_MOTION' in df.columns and 'ORBITAL_PERIOD' not in df.columns:
    df['ORBITAL_PERIOD'] = 86400 / df['MEAN_MOTION']
    print("Engineered 'ORBITAL_PERIOD'")

if 'MEAN_MOTION' in df.columns and 'SEMI_MAJOR_AXIS' not in df.columns:
    # Avoid division by zero or very small mean motion values
    # Check if MEAN_MOTION is not zero before calculating SEMI_MAJOR_AXIS
    non_zero_mean_motion_mask = (2 * np.pi * df['MEAN_MOTION'] / 86400)**2 != 0
    df['SEMI_MAJOR_AXIS'] = np.nan # Initialize with NaN
    df.loc[non_zero_mean_motion_mask, 'SEMI_MAJOR_AXIS'] = (MU / ((2 * np.pi * df.loc[non_zero_mean_motion_mask, 'MEAN_MOTION'] / 86400)**2))**(1/3)
    print("Engineered 'SEMI_MAJOR_AXIS'")


if 'APOAPSIS' in df.columns and 'SEMI_MAJOR_AXIS' in df.columns and 'ORBITAL_VELOCITY' not in df.columns:
     # Ensure the term inside the square root is non-negative
    arg = MU * (2 / (df['APOAPSIS'] + R_EARTH) - 1 / df['SEMI_MAJOR_AXIS'])
    df['ORBITAL_VELOCITY'] = np.sqrt(arg.clip(lower=0)) # Use clip to handle potential negative values due to floating point issues
    print("Engineered 'ORBITAL_VELOCITY'")

if 'PERIAPSIS' in df.columns and 'PERIGEE_ALTITUDE' not in df.columns:
    df['PERIGEE_ALTITUDE'] = df['PERIAPSIS'] - R_EARTH
    print("Engineered 'PERIGEE_ALTITUDE'")

if 'APOAPSIS' in df.columns and 'APOAPSIS_ALTITUDE' not in df.columns:
    df['APOAPSIS_ALTITUDE'] = df['APOAPSIS'] - R_EARTH
    print("Engineered 'APOAPSIS_ALTITUDE'")

if 'BSTAR' in df.columns and 'is_debris' not in df.columns:
    df['is_debris'] = (df['BSTAR'] > 0.0001).astype(int)
    print("Engineered 'is_debris'")


print("Columns after feature engineering:", df.columns.tolist())


# --- Data Splitting (Time-based for LSTM, Random for RF) ---


# For Random Forest Regressor
# Ensure that the target column exists after potential engineering
if 'APOAPSIS_ALTITUDE' in df.columns:
    numerical_features = df.select_dtypes(include=['float64', 'int64']).drop(columns=['NORAD_CAT_ID', 'EPOCH_UNIX'], errors='ignore') # Use errors='ignore' to handle missing columns gracefully
    scaler_rf = StandardScaler() # Use a different scaler instance for RF
    scaled_features_rf = scaler_rf.fit_transform(numerical_features)
    scaled_df_rf = pd.DataFrame(scaled_features_rf, columns=numerical_features.columns)

    if 'APOAPSIS_ALTITUDE' in scaled_df_rf.columns:
        X = scaled_df_rf.drop(columns=['APOAPSIS_ALTITUDE', 'PERIGEE_ALTITUDE'], errors='ignore')
        y = scaled_df_rf['APOAPSIS_ALTITUDE']
        X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X, y, test_size=0.2, random_state=42)
        print("Data split for Random Forest complete.")
    else:
        print("Warning: 'APOAPSIS_ALTITUDE' not found in scaled features for Random Forest.")
else:
     print("Warning: 'APOAPSIS_ALTITUDE' not found for Random Forest regression target.")


# For LSTM
if 'EPOCH_UNIX' in df.columns:
    df_sorted = df.sort_values(by='EPOCH_UNIX')
    cutoff_epoch = 1754000000
    train_df = df_sorted[df_sorted['EPOCH_UNIX'] < cutoff_epoch].copy() # Use .copy() to avoid SettingWithCopyWarning
    test_df = df_sorted[df_sorted['EPOCH_UNIX'] >= cutoff_epoch].copy() # Use .copy() to avoid SettingWithCopyWarning

    features_lstm = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
    # Filter features_lstm to only include columns present in the dataframe
    available_features_lstm = [f for f in features_lstm if f in df.columns]
    print("LSTM features available:", available_features_lstm)

    if available_features_lstm:
        scaler_lstm = MinMaxScaler() # Use a different scaler instance for LSTM
        train_scaled_lstm = scaler_lstm.fit_transform(train_df[available_features_lstm])
        test_scaled_lstm = scaler_lstm.transform(test_df[available_features_lstm])
        n_features = len(available_features_lstm)
        n_outputs = len(available_features_lstm) # Predict all features in the sequence

        def create_sequences(data, n_steps):
            X, y = [], []
            for i in range(len(data) - n_steps):
                X.append(data[i:(i + n_steps), :])
                y.append(data[i + n_steps, :]) # Predict the next step for all features
            return np.array(X), np.array(y)

        n_steps = 10
        if len(train_scaled_lstm) >= n_steps and len(test_scaled_lstm) >= n_steps:
            X_train_lstm, y_train_lstm = create_sequences(train_scaled_lstm, n_steps)
            X_test_lstm, y_test_lstm = create_sequences(test_scaled_lstm, n_steps)
            print("Data split and sequence creation for LSTM complete.")
        else:
            print(f"Warning: Not enough data to create sequences for LSTM (need at least {n_steps} steps).")
            X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None # Set to None if not enough data

    else:
        print("Error: No available features for LSTM training.")
        X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None # Set to None if no features

else:
    print("Error: 'EPOCH_UNIX' column not found for time-based splitting for LSTM.")
    train_df, test_df = None, None
    X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None


print("Data preparation complete.")

Successfully loaded space_debris_ml_ready.csv
Original columns: ['NORAD_CAT_ID', 'EPOCH_UNIX', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR']
Engineered 'ORBITAL_PERIOD'
Engineered 'SEMI_MAJOR_AXIS'
Engineered 'ORBITAL_VELOCITY'
Engineered 'PERIGEE_ALTITUDE'
Engineered 'APOAPSIS_ALTITUDE'
Engineered 'is_debris'
Columns after feature engineering: ['NORAD_CAT_ID', 'EPOCH_UNIX', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE', 'is_debris']
Data split for Random Forest complete.
LSTM features available: ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
Data split and sequence creation for LSTM co

In [5]:
# --- Phase 2: Random Forest and LSTM Models ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# --- Random Forest Regressor ---
print("\nTraining and evaluating the Random Forest Regressor...")
model_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train_rf, y_train_rf)
y_pred_rf = model_rf.predict(X_test_rf)
mse_rf = mean_squared_error(y_test_rf, y_pred_rf)
r_squared_rf = r2_score(y_test_rf, y_pred_rf)
mae_rf = mean_absolute_error(y_test_rf, y_pred_rf)
print(f'Random Forest R-squared: {r_squared_rf:.4f}')

# --- LSTM Model ---
print("\nTraining and evaluating the LSTM model...")
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:(i + n_steps), :])
        y.append(data[i + n_steps, :])
    return np.array(X), np.array(y)

features_lstm = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
scaler_lstm = MinMaxScaler()
train_scaled_lstm = scaler_lstm.fit_transform(train_df[features_lstm])
test_scaled_lstm = scaler_lstm.transform(test_df[features_lstm])
n_steps = 10
X_train_lstm, y_train_lstm = create_sequences(train_scaled_lstm, n_steps)
X_test_lstm, y_test_lstm = create_sequences(test_scaled_lstm, n_steps)
n_features = len(features_lstm)
n_outputs = len(features_lstm)
model_lstm = Sequential()
model_lstm.add(LSTM(50, activation='relu', input_shape=(n_steps, n_features)))
model_lstm.add(Dense(n_outputs))
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train_lstm, y_train_lstm, epochs=20, verbose=1)
y_pred_lstm = model_lstm.predict(X_test_lstm)
mse_lstm = mean_squared_error(y_test_lstm, y_pred_lstm)

print(f"\nLSTM Mean Squared Error: {mse_lstm:.4f}")


Training and evaluating the Random Forest Regressor...
Random Forest R-squared: 0.9999

Training and evaluating the LSTM model...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


242/242 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.0886
Epoch 2/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0301
Epoch 3/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0292
Epoch 4/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0297
Epoch 5/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0299
Epoch 6/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0287
Epoch 7/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0294
Epoch 8/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0292
Epoch 9/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0303
Epoch 10/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0309
Epoch 11/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0288
Epoch 12/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0303
Epoch 13/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0299
Epoch 14/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0289
Epoch 15/20
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0286


In [6]:
# --- Phase 3: 3D Visualization ---
import plotly.graph_objects as go
import numpy as np
import pandas as pd

print("\nGenerating 3D Visualization (simplified)...")

# Read the main dataframe - assuming the updated df from the previous cell is available
# If running this cell independently, you might need to reload the data
# try:
#     df = pd.read_csv('space_debris_ml_ready.csv') # <--- Update this if your new file has a different name
# except FileNotFoundError:
#     print("Error: 'space_debris_ml_ready.csv' not found. Please upload it.")
#     # exit() # Uncomment this line to stop execution if the file is not found
#     df = None # Set df to None if file not found to prevent further errors

if df is not None and 'SEMI_MAJOR_AXIS' in df.columns and 'is_debris' in df.columns:
    # Prepare for plotting
    # Ensure 'is_debris' is calculated if not already present
    if 'is_debris' not in df.columns and 'BSTAR' in df.columns:
         df['is_debris'] = (df['BSTAR'] > 0.0001).astype(int)
         print("Engineered 'is_debris' for visualization.")
    elif 'is_debris' not in df.columns:
         print("Warning: 'is_debris' column not available for visualization.")


    # Use engineered features for a simplified 3D representation
    # We can use SEMI_MAJOR_AXIS to represent a characteristic orbital radius
    # For a simplified visualization, let's create points on a sphere with radius equal to SEMI_MAJOR_AXIS
    # This is a placeholder and does not represent the true orbital path

    # Calculate a simplified position based on semi-major axis and a hypothetical position
    # We will use random angles for this simplified visualization to show something in 3D
    np.random.seed(42) # for reproducibility
    num_points_to_plot = 1000

    # Select relevant columns and take the head
    # Ensure columns exist before selecting
    cols_to_plot = ['SEMI_MAJOR_AXIS', 'is_debris']
    available_cols_to_plot = [col for col in cols_to_plot if col in df.columns]

    if len(available_cols_to_plot) == len(cols_to_plot):
        df_plot_subset = df[available_cols_to_plot].head(num_points_to_plot).copy()

        r = df_plot_subset['SEMI_MAJOR_AXIS']
        theta = np.random.rand(len(r)) * 2 * np.pi # Random angle
        phi = np.random.rand(len(r)) * np.pi # Random angle

        x = r * np.sin(phi) * np.cos(theta)
        y = r * np.sin(phi) * np.sin(theta)
        z = r * np.cos(phi)

        df_plot = pd.DataFrame({'x': x, 'y': y, 'z': z, 'is_debris': df_plot_subset['is_debris']})

        # Create the 3D plot
        fig = go.Figure(data=[go.Scatter3d(x=df_plot['x'], y=df_plot['y'], z=df_plot['z'], mode='markers',
                                        marker=dict(size=2, color=df_plot['is_debris'], colorscale='Viridis', opacity=0.8))])
        fig.update_layout(title='Simplified 3D Visualization of Orbits (based on Semi-Major Axis)',
                        scene=dict(xaxis_title='X (km)', yaxis_title='Y (km)', zaxis_title='Z (km)'),
                        margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
    else:
        missing = [col for col in cols_to_plot if col not in df.columns]
        print(f"Error: Cannot generate 3D plot. Missing required columns: {missing}")

else:
    print("Error: Dataframe not loaded or required columns ('SEMI_MAJOR_AXIS', 'is_debris') not available for 3D plot.")


Generating 3D Visualization (simplified)...


## Summary:

### Data Analysis Key Findings

*   A sample pandas DataFrame `orbits_df` was successfully generated, containing simulated 3D orbital data for 3 distinct objects over 100 timepoints.
*   An interactive 3D Plotly figure was created, including a central sphere representing Earth with a radius of 6371 km.
*   Separate 3D line traces were added to the plot for each simulated object's orbit, color-coded using Plotly's qualitative color palette.
*   Hover information was configured for each orbital trace to display the object ID, timestamp, and coordinates (x, y, z) when the user hovers over the lines.
*   The plot layout was updated to include the title "Interactive 3D Orbit Visualization", apply the 'plotly_dark' template, and ensure equal aspect ratios for the x, y, and z axes.

### Insights or Next Steps

*   The current simulation uses simplified trigonometric paths; future steps could involve implementing more realistic orbital mechanics (e.g., Keplerian orbits) for the simulation data.
*   Consider adding interactive elements like a time slider to animate the orbits or allow filtering by object ID for more detailed analysis.

## Summary:

### Data Analysis Key Findings

*   The `generate_keplerian_orbit` function was successfully implemented to calculate ECI coordinates from Keplerian elements using orbital mechanics principles.
*   Orbital data for a sample Low Earth Orbit (LEO) object with a semi-major axis of approximately 6771 km and for a sample Geostationary Orbit (GEO) object with a semi-major axis of approximately 42164 km were generated.
*   The generated orbital data for both LEO and GEO objects were successfully combined into a single pandas DataFrame named `realistic_orbits_df`.
*   An interactive 3D visualization was created using Plotly, successfully plotting the calculated orbits of both the LEO and GEO objects and including a sphere representing Earth.

### Insights or Next Steps

*   The current simulation assumes a single orbit under ideal Keplerian conditions. A valuable next step would be to extend the `generate_keplerian_orbit` function or add subsequent processing to simulate multiple orbits and incorporate orbital perturbations (e.g., atmospheric drag for LEO, solar radiation pressure, gravitational effects from other bodies) for a more realistic long-term trajectory.
*   Enhance the visualization by adding features like animating the orbits over time, displaying the current position of the satellites along the orbit, or adding labels for perigee and apogee points.

In [51]:
# Display the plot
fig.show()

print("Figure displayed.")

Figure displayed.


In [61]:
# Display the plot
fig.show()

print("Figure displayed.")

Figure displayed.


In [60]:
# Update layout for better visualization
fig.update_layout(
    title='Propagated Debris Orbits Visualization',
    template='plotly_dark', # Use dark theme
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data' # Ensure equal aspect ratios
    ),
    margin=dict(l=0, r=0, b=0, t=40) # Adjust margins
)

print("Plot layout updated.")

Plot layout updated.


In [59]:
import plotly.colors

# Add traces for each object's propagated orbit
unique_object_ids = df_propagated_orbits['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = df_propagated_orbits[df_propagated_orbits['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Propagated orbit traces added to the figure.")

Propagated orbit traces added to the figure.


In [58]:
# Add a sphere representing Earth using go.Surface
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure using go.Surface.")

Earth sphere added to the figure using go.Surface.


In [57]:
import plotly.graph_objects as go

# Create a new empty figure
fig = go.Figure()

print("Plotly figure object created.")

Plotly figure object created.


In [56]:
import pandas as pd

# Create a pandas DataFrame from the propagated orbit data
df_propagated_orbits = pd.DataFrame(propagated_orbits_data)

print("DataFrame for propagated orbits created.")
display(df_propagated_orbits.head())

DataFrame for propagated orbits created.


,object_id,timestamp,x_km,y_km,z_km
0,Debris_1,2025-09-16 18:32:24+00:00,4325.888866,-4697.318052,4641.859673
1,Debris_1,2025-09-16 18:42:24+00:00,7126.883673,-2680.533227,2200.878957
2,Debris_1,2025-09-16 18:52:24+00:00,7923.807807,92.110017,-862.531346
3,Debris_1,2025-09-16 19:02:24+00:00,6526.135560,2841.632602,-3689.194537
4,Debris_1,2025-09-16 19:12:24+00:00,3350.586593,4819.711128,-5511.713074


In [62]:
from skyfield.api import EarthSatellite, load
import datetime
import pytz
import numpy as np
import pandas as pd # Ensure pandas is imported for DataFrame creation

# Define a time scale
ts = load.timescale()

# Define the time range for propagation (e.g., the next 24 hours)
start_time_utc = datetime.datetime.now(pytz.utc)
end_time_utc = start_time_utc + datetime.timedelta(hours=24)
time_step_minutes = 10 # Define a reasonable time step

# Generate a sequence of time points
# Calculate the total number of steps
total_seconds = (end_time_utc - start_time_utc).total_seconds()
time_step_seconds = time_step_minutes * 60
total_steps = int(total_seconds / time_step_seconds)

# Create a range of timestamps in seconds from the start time
time_seconds = np.arange(0, total_steps * time_step_seconds, time_step_seconds)
# Convert the start time to a skyfield Time object
start_time_skyfield = ts.utc(start_time_utc.year, start_time_utc.month, start_time_utc.day,
                             start_time_utc.hour, start_time_utc.minute, start_time_utc.second)
# Add the time increments to the start time to get a series of skyfield Time objects
times = start_time_skyfield + time_seconds / (24 * 60 * 60) # Convert seconds to days for skyfield arithmetic


# Propagate orbits for the selected debris TLEs
propagated_orbits_data = []

# Increase the number of TLEs to select for broader analysis
num_tles_to_select_expanded = 100 # Propagate 100 debris objects
if len(tle_list_from_csv) < num_tles_to_select_expanded:
    num_tles_to_select_expanded = len(tle_list_from_csv)
    print(f"Warning: Only {len(tle_list_from_csv)} TLEs available in the list. Propagating all available TLEs.")

selected_debris_tles_expanded = tle_list_from_csv[:num_tles_to_select_expanded]


print(f"\nPropagating orbits for {len(selected_debris_tles_expanded)} debris objects...")

for i, tle_pair in enumerate(selected_debris_tles_expanded):
    line1, line2 = tle_pair
    try:
        # Create an EarthSatellite object for each TLE pair
        satellite = EarthSatellite(line1, line2, f'Debris_{i+1}')

        # Propagate the satellite's position over the defined time range
        position = satellite.at(times)

        # Get ECI coordinates (x, y, z) in kilometers
        x_km, y_km, z_km = position.position.km

        # Store the propagated data for this object
        for j in range(len(times)):
            propagated_orbits_data.append({
                'object_id': f'Debris_{i+1}',
                'timestamp': times[j].utc_datetime(), # Store as datetime objects
                'x_km': x_km[j],
                'y_km': y_km[j],
                'z_km': z_km[j]
            })
    except Exception as e:
        print(f"Error propagating TLE set {i+1}: {e}")
        # Continue to the next TLE set even if one fails

print("Orbit propagation complete.")

# The propagated_orbits_data list now contains the data for all selected objects
# The next step in the plan is to create a DataFrame from this data.


Propagating orbits for 100 debris objects...
Orbit propagation complete.


In [54]:
# Select a subset of TLEs (e.g., the first 20 debris TLEs)
num_tles_to_select = 20
selected_debris_tles = tle_list_from_csv[:num_tles_to_select]

print(f"Selected {len(selected_debris_tles)} debris TLE sets.")
# Optional: Print the first selected TLE set to verify
# if selected_debris_tles:
#     print("\nFirst selected TLE set:")
#     print(selected_debris_tles[0][0])
#     print(selected_debris_tles[0][1])

Selected 20 debris TLE sets.


## Summary:

### Data Analysis Key Findings

*   A Gradient Boosting Classifier was successfully initialized and trained on the prepared training data (`X_train`, `y_train`).
*   The model's performance was evaluated on the test data (`X_test`, `y_test`) using a classification report and confusion matrix.
*   The classification report shows excellent precision, recall, and f1-scores for both the high-risk (1) and low-risk (0) classes, indicating that the model is highly accurate in classifying space objects based on the defined risk criterion and features.
*   The confusion matrix confirms the high accuracy, showing a very low number of misclassifications (only 1 false positive and 0 false negatives in this specific test set).

### Insights or Next Steps

*   The Gradient Boosting Classifier appears to be a very effective model for this binary high-risk classification task based on the provided dataset and features.
*   Comparing the performance metrics (classification report, confusion matrix, and potentially others like AUC) of the Gradient Boosting model to the Random Forest model trained earlier would provide valuable insights into which model is better suited for this specific classification problem.
*   Further analysis could involve investigating the features that are most important for the Gradient Boosting model's predictions.

In [53]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_gb = model_gb_classifier.predict(X_test)

print("\nGradient Boosting Classifier - Classification Report:")
print(classification_report(y_test, y_pred_gb))

print("\nGradient Boosting Classifier - Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))


Gradient Boosting Classifier - Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       100
           1       1.00      1.00      1.00      2801

    accuracy                           1.00      2901
   macro avg       1.00      0.99      1.00      2901
weighted avg       1.00      1.00      1.00      2901


Gradient Boosting Classifier - Confusion Matrix:
[[  99    1]
 [   0 2801]]


In [52]:
from sklearn.ensemble import GradientBoostingClassifier

# Initialize and train the GradientBoostingClassifier
model_gb_classifier = GradientBoostingClassifier(random_state=42)
model_gb_classifier.fit(X_train, y_train)
print("GradientBoostingClassifier model trained.")

GradientBoostingClassifier model trained.


In [49]:
# Define the list of timestamps for the slider steps
timestamps = orbits_df['timestamp'].unique()

# Create slider steps
slider_steps = []
for i, timestamp in enumerate(timestamps):
    label = timestamp.strftime('%Y-%m-%d %H:%M') # Format timestamp for display
    step = dict(
        method='animate',
        args=[[str(timestamp)], dict(mode='immediate', frame=dict(duration=100, redraw=True), transition=dict(duration=0))],
        label=label
    )
    slider_steps.append(step)

# Configure the slider
sliders = [dict(
    active=0,
    yanchor='top',
    xanchor='left',
    currentvalue=dict(font=dict(size=16), prefix='Time: ', visible=True, xanchor='right'),
    transition=dict(duration=300, easing='cubic-in-out'),
    pad=dict(b=10, t=50),
    len=0.9,
    x=0.1,
    y=0,
    steps=slider_steps
)]

# Configure the play/pause buttons
updatemenus = [dict(
    type='buttons',
    buttons=[dict(label='Play',
                  method='animate',
                  args=[None, dict(frame=dict(duration=100, redraw=True), transition=dict(duration=0), fromcurrent=True, mode='immediate')]),
             dict(label='Pause',
                  method='animate',
                  args=[[None], dict(frame=dict(duration=0, redraw=True), mode='immediate')])],
    direction='left',
    pad=dict(r=10, t=87),
    showactive=False,
    x=0.1,
    y=0,
    xanchor='right',
    yanchor='top'
)]

# Update the figure layout with the slider and buttons
fig.update_layout(
    updatemenus=updatemenus,
    sliders=sliders
)

print("Figure layout configured with slider and buttons.")

Figure layout configured with slider and buttons.


In [48]:
import plotly.graph_objects as go

# Create animation frames
frames = []
for timestamp in orbits_df['timestamp'].unique():
    frame_data = []
    # Filter data for the current timestamp
    df_current_time = orbits_df[orbits_df['timestamp'] == timestamp]

    # Add a trace for each object at the current timestamp
    for obj_id in df_current_time['object_id'].unique():
        obj_df_current_time = df_current_time[df_current_time['object_id'] == obj_id]
        frame_data.append(go.Scatter3d(
            x=obj_df_current_time['x_km'],
            y=obj_df_current_time['y_km'],
            z=obj_df_current_time['z_km'],
            mode='markers', # Use markers for the animated points
            name=obj_id,
            marker=dict(size=5), # Use a consistent marker size
             # Hover info will be handled by the base trace, or can be added here if needed per frame
        ))
    frames.append(go.Frame(data=frame_data, name=str(timestamp))) # Use timestamp as frame name

print(f"Created {len(frames)} animation frames.")

Created 100 animation frames.


In [50]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import plotly.colors

# Initialize an empty figure
fig = go.Figure()

# Add the Earth sphere (static)
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

# Add traces for each object's orbit at the first timestamp
unique_object_ids = orbits_df['object_id'].unique()
colors = plotly.colors.qualitative.Plotly # Define a color palette

# Get data for the first timestamp
initial_timestamp = orbits_df['timestamp'].min()
orbits_df_initial = orbits_df[orbits_df['timestamp'] == initial_timestamp]

for i, obj_id in enumerate(unique_object_ids):
    obj_df_initial = orbits_df_initial[orbits_df_initial['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df_initial['x_km'],
        y=obj_df_initial['y_km'],
        z=obj_df_initial['z_km'],
        mode='markers', # Use markers to represent the initial position
        name=f'{obj_id} (Initial)',
        marker=dict(color=colors[i % len(colors)], size=5),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df_initial.iterrows()]
    ))

# Set initial layout (can be updated later)
fig.update_layout(
    title='Interactive 3D Orbit Animation',
    template='plotly_dark',
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.frames = frames # Add the generated frames to the figure

print("Frames added to the figure.")

Frames added to the figure.


In [46]:
# Display the plot
fig.show()

print("Figure displayed.")

Figure displayed.


In [45]:
# Display the plot
fig.show()

print("Figure displayed.")

Figure displayed.


In [44]:
# Update layout for better visualization
fig.update_layout(
    title='Interactive 3D Orbit Visualization',
    template='plotly_dark', # Use dark theme
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data' # Ensure equal aspect ratios
    ),
    margin=dict(l=0, r=0, b=0, t=40) # Adjust margins
)

print("Plot layout updated.")

Plot layout updated.


In [43]:
import plotly.colors

# Add traces for each object's orbit
unique_object_ids = realistic_orbits_df['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = realistic_orbits_df[realistic_orbits_df['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Orbital traces added to the figure.")

Orbital traces added to the figure.


In [42]:
# Add a sphere representing Earth using go.Surface
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure using go.Surface.")

Earth sphere added to the figure using go.Surface.


In [41]:
import plotly.graph_objects as go

# Create an empty figure
fig = go.Figure()

print("Plotly figure object created.")

Plotly figure object created.


In [40]:
import pandas as pd
import numpy as np

# --- Generate Orbital Data ---

# Define Keplerian elements for a sample LEO object
# Approximate values for ISS or similar LEO
leo_semi_major_axis = 6371.0 + 400.0  # Earth radius + ~400 km altitude
leo_eccentricity = 0.0005
leo_inclination_deg = 51.6
leo_raan_deg = 250.0
leo_arg_of_perigee_deg = 90.0
leo_true_anomaly_start_deg = 0.0

# Define Keplerian elements for a sample GEO object
# Approximate values for GEO
geo_semi_major_axis = 42164.0 # ~35786 km altitude + Earth radius
geo_eccentricity = 0.0001
geo_inclination_deg = 0.01 # Ideally close to 0 for GEO
geo_raan_deg = 0.0
geo_arg_of_perigee_deg = 0.0
geo_true_anomaly_start_deg = 0.0

# Generate data for LEO object
x_leo, y_leo, z_leo = generate_keplerian_orbit(
    semi_major_axis=leo_semi_major_axis,
    eccentricity=leo_eccentricity,
    inclination_deg=leo_inclination_deg,
    raan_deg=leo_raan_deg,
    arg_of_perigee_deg=leo_arg_of_perigee_deg,
    true_anomaly_start_deg=leo_true_anomaly_start_deg,
    num_points=200 # More points for smoother visualization
)

# Generate data for GEO object
x_geo, y_geo, z_geo = generate_keplerian_orbit(
    semi_major_axis=geo_semi_major_axis,
    eccentricity=geo_eccentricity,
    inclination_deg=geo_inclination_deg,
    raan_deg=geo_raan_deg,
    arg_of_perigee_deg=geo_arg_of_perigee_deg,
    true_anomaly_start_deg=geo_true_anomaly_start_deg,
    num_points=200 # More points for smoother visualization
)

# Combine data into a single DataFrame
leo_df = pd.DataFrame({
    'object_id': 'LEO_Object',
    'timestamp': pd.to_datetime(np.linspace(0, 24*3600, len(x_leo)), unit='s'), # Simulate timestamps over a period
    'x_km': x_leo,
    'y_km': y_leo,
    'z_km': z_leo
})

geo_df = pd.DataFrame({
    'object_id': 'GEO_Object',
    'timestamp': pd.to_datetime(np.linspace(0, 24*3600*365, len(x_geo)), unit='s'), # Simulate timestamps over a longer period for GEO relative motion
    'x_km': x_geo,
    'y_km': y_geo,
    'z_km': z_geo
})

realistic_orbits_df = pd.concat([leo_df, geo_df], ignore_index=True)

print("Generated orbital data for LEO and GEO objects.")
display(realistic_orbits_df.head())

Generated orbital data for LEO and GEO objects.


,object_id,timestamp,x_km,y_km,z_km
0,LEO_Object,1970-01-01 00:00:00.000000000,3950.175274,-1437.746220,5303.735205
1,LEO_Object,1970-01-01 00:07:14.170854271,4021.277940,-1236.270461,5301.093086
2,LEO_Object,1970-01-01 00:14:28.341708543,4088.374179,-1033.562774,5293.169356
3,LEO_Object,1970-01-01 00:21:42.512562814,4151.397199,-829.824918,5279.971888
4,LEO_Object,1970-01-01 00:28:56.683417085,4210.284255,-625.259677,5261.513799


In [39]:
import numpy as np

def generate_keplerian_orbit(semi_major_axis, eccentricity, inclination_deg, raan_deg, arg_of_perigee_deg, true_anomaly_start_deg, num_points=100):
    """
    Calculates Earth-Centered Inertial (ECI) coordinates for one orbit
    using Keplerian elements.

    Args:
        semi_major_axis (float): Semi-major axis in km.
        eccentricity (float): Eccentricity (unitless).
        inclination_deg (float): Inclination in degrees.
        raan_deg (float): Right Ascension of the Ascending Node in degrees.
        arg_of_perigee_deg (float): Argument of Perigee in degrees.
        true_anomaly_start_deg (float): Starting True Anomaly in degrees.
        num_points (int): Number of points to generate for the orbit.

    Returns:
        tuple: A tuple containing three numpy arrays (x_km, y_km, z_km)
               representing the ECI coordinates in kilometers over one orbit.
               Returns None if input is invalid.
    """
    # Basic validation
    if semi_major_axis <= 0 or eccentricity < 0:
        print("Invalid Keplerian elements.")
        return None

    # Convert angles to radians
    inclination_rad = np.deg2rad(inclination_deg)
    raan_rad = np.deg2rad(raan_deg)
    arg_of_perigee_rad = np.deg2rad(arg_of_perigee_deg)
    true_anomaly_start_rad = np.deg2rad(true_anomaly_start_deg)


    # Generate a range of true anomaly values for one orbit
    true_anomaly_range_rad = np.linspace(true_anomaly_start_rad, true_anomaly_start_rad + 2 * np.pi, num_points)

    # Calculate radial distance (r) for each true anomaly
    r = semi_major_axis * (1 - eccentricity**2) / (1 + eccentricity * np.cos(true_anomaly_range_rad))

    # Calculate position in the orbital plane (perifocal frame)
    x_perifocal = r * np.cos(true_anomaly_range_rad)
    y_perifocal = r * np.sin(true_anomaly_range_rad)
    z_perifocal = np.zeros_like(true_anomaly_range_rad)

    # Convert from perifocal frame to ECI frame using rotation matrices
    # Rotation 1: Rotate by argument of perigee around Z-axis
    # Rotation 2: Rotate by inclination around X-axis
    # Rotation 3: Rotate by RAAN around Z-axis

    # Combined rotation matrix (Z-X-Z Euler angles)
    cos_raan = np.cos(raan_rad)
    sin_raan = np.sin(raan_rad)
    cos_inc = np.cos(inclination_rad)
    sin_inc = np.sin(inclination_rad)
    cos_argp = np.cos(arg_of_perigee_rad)
    sin_argp = np.sin(arg_of_perigee_rad)

    # R_z(RAAN) * R_x(inc) * R_z(arg_of_perigee)
    # First column of the combined rotation matrix
    R11 = cos_raan * cos_argp - sin_raan * sin_argp * cos_inc
    R21 = sin_raan * cos_argp + cos_raan * sin_argp * cos_inc
    R31 = sin_argp * sin_inc

    # Second column of the combined rotation matrix
    R12 = -cos_raan * sin_argp - sin_raan * cos_argp * cos_inc
    R22 = -sin_raan * sin_argp + cos_raan * cos_argp * cos_inc
    R32 = cos_argp * sin_inc

    # Third column of the combined rotation matrix
    R13 = sin_raan * sin_inc
    R23 = -cos_raan * sin_inc
    R33 = cos_inc

    # Apply the rotation matrix to the perifocal coordinates
    x_eci = R11 * x_perifocal + R12 * y_perifocal + R13 * z_perifocal
    y_eci = R21 * x_perifocal + R22 * y_perifocal + R23 * z_perifocal
    z_eci = R31 * x_perifocal + R32 * y_perifocal + R33 * z_perifocal

    return x_eci, y_eci, z_eci

print("Updated generate_keplerian_orbit function with orbital mechanics.")

Updated generate_keplerian_orbit function with orbital mechanics.


In [30]:
# Display the plot
fig.show()

print("Figure displayed.")

Figure displayed.


In [29]:
# Update layout for better visualization
fig.update_layout(
    title='Interactive 3D Orbit Visualization',
    template='plotly_dark', # Use dark theme
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data' # Ensure equal aspect ratios
    ),
    margin=dict(l=0, r=0, b=0, t=40) # Adjust margins
)

print("Plot layout updated.")

Plot layout updated.


In [28]:
import plotly.colors

# Add traces for each object's orbit
unique_object_ids = orbits_df['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = orbits_df[orbits_df['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Orbital traces added to the figure.")

Orbital traces added to the figure.


In [27]:
# Add a sphere representing Earth using go.Surface
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure using go.Surface.")

Earth sphere added to the figure using go.Surface.


In [25]:
import plotly.graph_objects as go

# Create an empty figure
fig = go.Figure()

print("Plotly figure object created.")

Plotly figure object created.


In [24]:
import pandas as pd
import numpy as np

# List of object IDs
object_ids = ['object_1', 'object_2', 'object_3']

# Generate a sequence of timestamps
num_timepoints = 100
timestamps = pd.date_range(start='2023-01-01', periods=num_timepoints, freq='H')

# Simulate orbital paths for each object
all_orbits_data = []

for obj_id in object_ids:
    # Simulate orbital parameters
    amplitude_x = np.random.uniform(7000, 10000) # Varying semi-major axis
    amplitude_y = np.random.uniform(7000, 10000)
    amplitude_z = np.random.uniform(5000, 8000)
    frequency_x = np.random.uniform(0.1, 0.5) # Varying orbital period
    frequency_y = np.random.uniform(0.1, 0.5)
    frequency_z = np.random.uniform(0.1, 0.5)
    noise_level = np.random.uniform(50, 200) # Add some noise

    # Generate coordinates
    t = np.arange(num_timepoints)
    x_km = amplitude_x * np.cos(2 * np.pi * frequency_x * t / num_timepoints) + np.random.normal(0, noise_level, num_timepoints)
    y_km = amplitude_y * np.sin(2 * np.pi * frequency_y * t / num_timepoints) + np.random.normal(0, noise_level, num_timepoints)
    z_km = amplitude_z * np.sin(2 * np.pi * frequency_z * t / num_timepoints + np.pi/4) + np.random.normal(0, noise_level, num_timepoints) # Add phase shift for z

    # Create temporary DataFrame for the current object
    temp_df = pd.DataFrame({
        'object_id': obj_id,
        'timestamp': timestamps,
        'x_km': x_km,
        'y_km': y_km,
        'z_km': z_km
    })

    all_orbits_data.append(temp_df)

# Concatenate all temporary DataFrames
orbits_df = pd.concat(all_orbits_data, ignore_index=True)

# Print the head of the DataFrame
display(orbits_df.head())

/tmp/ipython-input-2233938007.py:9: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,object_id,timestamp,x_km,y_km,z_km
0,object_1,2023-01-01 00:00:00,8949.319846,-42.631654,3711.638869
1,object_1,2023-01-01 01:00:00,8842.409299,130.320782,3767.414975
2,object_1,2023-01-01 02:00:00,8934.175773,333.326153,3742.932908
3,object_1,2023-01-01 03:00:00,8881.851264,376.649841,3845.278246
4,object_1,2023-01-01 04:00:00,8869.512711,532.292915,3916.192162


In [16]:
import pandas as pd

# Load the space_debris_combined.csv dataset
try:
    tle_df_combined = pd.read_csv('/content/space_debris_combined.csv')
    print("Successfully loaded space_debris_combined.csv")
    print("Columns available:", tle_df_combined.columns.tolist())

    # Check if 'LINE1' and 'LINE2' columns exist
    if 'LINE1' in tle_df_combined.columns and 'LINE2' in tle_df_combined.columns:
        # Extract TLEs and format as a list of [line1, line2] pairs
        tle_list_from_csv = []
        for index, row in tle_df_combined.iterrows():
            # Ensure both lines are strings and not NaN
            line1 = str(row['LINE1']).strip() if pd.notna(row['LINE1']) else None
            line2 = str(row['LINE2']).strip() if pd.notna(row['LINE2']) else None

            if line1 and line2: # Only add if both lines are valid
                tle_list_from_csv.append([line1, line2])
            else:
                # Optional: Print a warning if a row is skipped due to missing TLE lines
                # print(f"Warning: Skipping row {index} due to missing TLE lines.")
                pass


        print(f"\nExtracted {len(tle_list_from_csv)} sets of TLEs from the CSV.")

        # You can now use the tle_list_from_csv for further processing
        # Example: Print the first few extracted TLE sets
        # print("\nFirst 5 extracted TLE sets:")
        # for i, tle_set in enumerate(tle_list_from_csv[:5]):
        #     print(f"Set {i+1}:")
        #     print(tle_set[0])
        #     print(tle_set[1])

    else:
        print("Error: 'LINE1' or 'LINE2' columns not found in the CSV for TLE extraction.")
        tle_list_from_csv = None

except FileNotFoundError:
    print("Error: 'space_debris_combined.csv' not found. Please ensure it's uploaded.")
    tle_list_from_csv = None

Successfully loaded space_debris_combined.csv
Columns available: ['NORAD_CAT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'COUNTRY_CODE', 'RCS_SIZE', 'NAME', 'LINE1', 'LINE2']

Extracted 9669 sets of TLEs from the CSV.


In [15]:
# Install the spacetrack library if you haven't already
!pip install spacetrack

from spacetrack import SpaceTrackClient
import warnings

def fetch_recent_debris_tles(username, password):
    """
    Fetches the 100 most recently updated debris TLEs from Space-Track.org.

    Args:
        username (str): Your Space-Track.org username.
        password (str): Your Space-Track.org password.

    Returns:
        list: A list of lists, where each inner list contains the two TLE
              lines for one object, or None if fetching fails.
              Example: [['LINE1_OBJ1', 'LINE2_OBJ1'], ['LINE1_OBJ2', 'LINE2_OBJ2'], ...]
    """
    st = None
    try:
        # Authenticate with Space-Track.org
        st = SpaceTrackClient(username, password)
        print("Successfully authenticated with Space-Track.org")

        # Query for the 100 most recently updated debris TLEs
        # orderby='EPOCH desc' gets the most recent TLEs first
        # limit=100 restricts to the top 100 results
        # OBJECT_TYPE='DEBRIS' filters for debris objects
        # format='tle' requests the data in TLE format
        # latest=True ensures the latest TLE for each object is returned (though orderby handles recency)
        print("Fetching recent debris TLEs...")
        data = st.tle(orderby='EPOCH desc', limit=100, object_type='DEBRIS', format='tle', latest=True)
        print("Successfully fetched TLE data.")

        # The 'data' variable is a string containing TLEs separated by newlines
        # We need to parse this string into a list of [line1, line2] pairs

        # Split the string into individual lines
        lines = data.strip().split('\n')

        # Ensure we have an even number of lines (TLEs come in pairs)
        if len(lines) % 2 != 0:
            warnings.warn("Fetched data has an odd number of lines, which is unexpected for TLEs.")
            # Optionally handle this error more strictly

        # Group lines into [line1, line2] pairs
        tle_list = []
        for i in range(0, len(lines) - 1, 2):
            tle_list.append([lines[i], lines[i+1]])

        print(f"Formatted {len(tle_list)} TLE sets.")
        return tle_list

    except Exception as e:
        print(f"An error occurred while fetching TLEs: {e}")
        return None

# --- Example Usage ---
# WARNING: Do NOT save your real username and password directly in this code.
# Use Colab's Secrets Manager or other secure methods for storing credentials.
# Replace 'YOUR_USERNAME' and 'YOUR_PASSWORD' with your actual credentials
# when running this in a secure environment.

# Example using placeholders (replace with secure method in practice)
# from google.colab import userdata
# username = userdata.get('SPACE_TRACK_USERNAME')
# password = userdata.get('SPACE_TRACK_PASSWORD')

# For demonstration with placeholders:
username_placeholder = 'YOUR_USERNAME'
password_placeholder = 'YOUR_PASSWORD'

recent_debris_tles = fetch_recent_debris_tles(username_placeholder, password_placeholder)

if recent_debris_tles:
    print(f"\nFetched {len(recent_debris_tles)} sets of recent debris TLEs.")
    # You can now process the recent_debris_tles list
    # Example: print the first TLE set
    # if recent_debris_tles:
    #     print("\nFirst fetched TLE set:")
    #     print(recent_debris_tles[0][0])
    #     print(recent_debris_tles[0][1])
else:
    print("\nFailed to fetch recent debris TLEs.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.4/511.4 kB 8.6 MB/s eta 0:00:00
Successfully authenticated with Space-Track.org
Fetching recent debris TLEs...
An error occurred while fetching TLEs: Client error '401 Unauthorized' for url 'https://www.space-track.org/ajaxauth/login'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401
Space-Track response:
{"Login":"Password does not meet minimum length requirements."}

Failed to fetch recent debris TLEs.


In [14]:
# Install skyfield if you haven't already
!pip install skyfield pytz pandas --upgrade

# Import necessary libraries
from skyfield.api import EarthSatellite, Topos, load
import datetime
import pytz
import pandas as pd
import numpy as np

# Load the TLE data for the International Space Station
line1 = '1 25544U 98067A   25259.54879574  .00016717  00000-0  30627-3 0  9992'
line2 = '2 25544  51.6416 254.1175 0006753 124.7176 235.3852 15.49511149234249'

# Create an EarthSatellite object
satellite = EarthSatellite(line1, line2, 'ISS (ZARYA)')

# Define a time scale
ts = load.timescale()

# Define the time range for the next 24 hours
# Start from the current UTC time
start_time_utc = datetime.datetime.now(pytz.utc)
# Define the end time (24 hours from the start time)
end_time_utc = start_time_utc + datetime.timedelta(hours=24)
# Define the time step (every 5 minutes)
time_step_minutes = 5

# Generate a sequence of time points using numpy and convert to skyfield Time objects
# Calculate the total number of steps
total_steps = int((end_time_utc - start_time_utc).total_seconds() / (time_step_minutes * 60))
# Create a range of timestamps in seconds from the start time
time_seconds = np.arange(0, total_steps * time_step_minutes * 60, time_step_minutes * 60)
# Convert the start time to a skyfield Time object
start_time_skyfield = ts.utc(start_time_utc.year, start_time_utc.month, start_time_utc.day,
                             start_time_utc.hour, start_time_utc.minute, start_time_utc.second)
# Add the time increments to the start time
times = start_time_skyfield + time_seconds / (24 * 60 * 60) # Convert seconds to days for skyfield arithmetic


# Propagate the satellite's position using the SGP4 model over the time range
# The .at() method returns a Position object
position = satellite.at(times)

# Calculate the Earth-Centered Inertial (ECI) coordinates (x, y, z) in kilometers
# The .position.km attribute gives the ECI coordinates in kilometers
x_km, y_km, z_km = position.position.km

# Store the results (timestamp, x_km, y_km, z_km) in a new pandas DataFrame
# Create a list of dictionaries for the DataFrame
data = []
for i in range(len(times)):
    data.append({
        'timestamp': times[i].utc_datetime(), # Store as datetime objects
        'x_km': x_km[i],
        'y_km': y_km[i],
        'z_km': z_km[i]
    })

# Create the pandas DataFrame
df_orbit = pd.DataFrame(data)

# Display the first 5 rows of the DataFrame
display(df_orbit.head())

,timestamp,x_km,y_km,z_km
0,2025-09-16 17:44:05+00:00,-3194.418493,-5761.491727,-1695.127406
1,2025-09-16 17:49:05+00:00,-1955.424093,-6511.890232,73.351147
2,2025-09-16 17:54:05+00:00,-495.135198,-6525.215643,1833.457659
3,2025-09-16 17:59:05+00:00,1021.234773,-5799.197424,3385.210030
4,2025-09-16 18:04:05+00:00,2421.832011,-4415.804170,4552.129667


**Note:** Before running the code below, make sure you have stored your Space-Track.org username and password in Colab's Secrets Manager with the keys `SPACE_TRACK_USERNAME` and `SPACE_TRACK_PASSWORD`.

In [4]:
# --- Data Loading and Preprocessing ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# You will need to upload 'space_debris_ml_ready.csv' to your Colab session.
# Assuming the new dataset is named 'space_debris_new.csv' or similar,
# please update the filename below if necessary.
file_path = 'space_debris_ml_ready.csv' # <--- Update this if your new file has a different name

try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    print("Original columns:", df.columns.tolist())

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    print("Please ensure the file is uploaded to your Colab session.")
    exit()

# Engineering features from the original dataset
# Ensure that the necessary columns for feature engineering exist in the new dataset
required_cols = ['MEAN_MOTION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'NORAD_CAT_ID', 'EPOCH_UNIX', 'ECCENTRICITY', 'INCLINATION']
if not all(col in df.columns for col in required_cols):
    missing = [col for col in required_cols if col not in df.columns]
    print(f"Error: The dataset is missing required columns for feature engineering: {missing}")
    # Attempt to proceed with available columns, but be aware of potential issues
    # exit() # Uncomment this line to stop execution if critical columns are missing

R_EARTH = 6371.0
MU = 398600.4418

# Re-calculate engineered features to ensure they are present
if 'MEAN_MOTION' in df.columns and 'ORBITAL_PERIOD' not in df.columns:
    df['ORBITAL_PERIOD'] = 86400 / df['MEAN_MOTION']
    print("Engineered 'ORBITAL_PERIOD'")

if 'MEAN_MOTION' in df.columns and 'SEMI_MAJOR_AXIS' not in df.columns:
    # Avoid division by zero or very small mean motion values
    # Check if MEAN_MOTION is not zero before calculating SEMI_MAJOR_AXIS
    non_zero_mean_motion_mask = (2 * np.pi * df['MEAN_MOTION'] / 86400)**2 != 0
    df['SEMI_MAJOR_AXIS'] = np.nan # Initialize with NaN
    df.loc[non_zero_mean_motion_mask, 'SEMI_MAJOR_AXIS'] = (MU / ((2 * np.pi * df.loc[non_zero_mean_motion_mask, 'MEAN_MOTION'] / 86400)**2))**(1/3)
    print("Engineered 'SEMI_MAJOR_AXIS'")


if 'APOAPSIS' in df.columns and 'SEMI_MAJOR_AXIS' in df.columns and 'ORBITAL_VELOCITY' not in df.columns:
     # Ensure the term inside the square root is non-negative
    arg = MU * (2 / (df['APOAPSIS'] + R_EARTH) - 1 / df['SEMI_MAJOR_AXIS'])
    df['ORBITAL_VELOCITY'] = np.sqrt(arg.clip(lower=0)) # Use clip to handle potential negative values due to floating point issues
    print("Engineered 'ORBITAL_VELOCITY'")

if 'PERIAPSIS' in df.columns and 'PERIGEE_ALTITUDE' not in df.columns:
    df['PERIGEE_ALTITUDE'] = df['PERIAPSIS'] - R_EARTH
    print("Engineered 'PERIGEE_ALTITUDE'")

if 'APOAPSIS' in df.columns and 'APOAPSIS_ALTITUDE' not in df.columns:
    df['APOAPSIS_ALTITUDE'] = df['APOAPSIS'] - R_EARTH
    print("Engineered 'APOAPSIS_ALTITUDE'")

if 'BSTAR' in df.columns and 'is_debris' not in df.columns:
    df['is_debris'] = (df['BSTAR'] > 0.0001).astype(int)
    print("Engineered 'is_debris'")


print("Columns after feature engineering:", df.columns.tolist())


# --- Data Splitting (Time-based for LSTM, Random for RF) ---


# For Random Forest Regressor
# Ensure that the target column exists after potential engineering
if 'APOAPSIS_ALTITUDE' in df.columns:
    numerical_features = df.select_dtypes(include=['float64', 'int64']).drop(columns=['NORAD_CAT_ID', 'EPOCH_UNIX'], errors='ignore') # Use errors='ignore' to handle missing columns gracefully
    scaler_rf = StandardScaler() # Use a different scaler instance for RF
    scaled_features_rf = scaler_rf.fit_transform(numerical_features)
    scaled_df_rf = pd.DataFrame(scaled_features_rf, columns=numerical_features.columns)

    if 'APOAPSIS_ALTITUDE' in scaled_df_rf.columns:
        X = scaled_df_rf.drop(columns=['APOAPSIS_ALTITUDE', 'PERIGEE_ALTITUDE'], errors='ignore')
        y = scaled_df_rf['APOAPSIS_ALTITUDE']
        X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(X, y, test_size=0.2, random_state=42)
        print("Data split for Random Forest complete.")
    else:
        print("Warning: 'APOAPSIS_ALTITUDE' not found in scaled features for Random Forest.")
else:
     print("Warning: 'APOAPSIS_ALTITUDE' not found for Random Forest regression target.")


# For LSTM
if 'EPOCH_UNIX' in df.columns:
    df_sorted = df.sort_values(by='EPOCH_UNIX')
    # Increased cutoff_epoch to include more data for LSTM training
    cutoff_epoch = df_sorted['EPOCH_UNIX'].quantile(0.8) # Use 80% of data for training
    train_df = df_sorted[df_sorted['EPOCH_UNIX'] < cutoff_epoch].copy() # Use .copy() to avoid SettingWithCopyWarning
    test_df = df_sorted[df_sorted['EPOCH_UNIX'] >= cutoff_epoch].copy() # Use .copy() to avoid SettingWithCopyWarning

    features_lstm = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
    # Filter features_lstm to only include columns present in the dataframe
    available_features_lstm = [f for f in features_lstm if f in df.columns]
    print("LSTM features available:", available_features_lstm)

    if available_features_lstm:
        scaler_lstm = MinMaxScaler() # Use a different scaler instance for LSTM
        # Check if train_df and test_df are not empty before scaling
        if not train_df.empty and not test_df.empty:
            train_scaled_lstm = scaler_lstm.fit_transform(train_df[available_features_lstm])
            test_scaled_lstm = scaler_lstm.transform(test_df[available_features_lstm])
            n_features = len(available_features_lstm)
            n_outputs = len(available_features_lstm) # Predict all features in the sequence

            def create_sequences(data, n_steps):
                X, y = [], []
                for i in range(len(data) - n_steps):
                    X.append(data[i:(i + n_steps), :])
                    y.append(data[i + n_steps, :]) # Predict the next step for all features
                return np.array(X), np.array(y)

            n_steps = 10
            if len(train_scaled_lstm) >= n_steps and len(test_scaled_lstm) >= n_steps:
                X_train_lstm, y_train_lstm = create_sequences(train_scaled_lstm, n_steps)
                X_test_lstm, y_test_lstm = create_sequences(test_scaled_lstm, n_steps)
                print("Data split and sequence creation for LSTM complete.")
            else:
                print(f"Warning: Not enough data to create sequences for LSTM (need at least {n_steps} steps).")
                X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None # Set to None if not enough data
        else:
            print("Warning: Training or testing dataframe for LSTM is empty after time-based splitting.")
            X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None # Set to None if not enough data


    else:
        print("Error: No available features for LSTM training.")
        X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None # Set to None if no features

else:
    print("Error: 'EPOCH_UNIX' column not found for time-based splitting for LSTM.")
    train_df, test_df = None, None
    X_train_lstm, y_train_lstm, X_test_lstm, y_test_lstm = None, None, None, None


print("Data preparation complete.")

Successfully loaded space_debris_ml_ready.csv
Original columns: ['NORAD_CAT_ID', 'EPOCH_UNIX', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR']
Engineered 'ORBITAL_PERIOD'
Engineered 'SEMI_MAJOR_AXIS'
Engineered 'ORBITAL_VELOCITY'
Engineered 'PERIGEE_ALTITUDE'
Engineered 'APOAPSIS_ALTITUDE'
Engineered 'is_debris'
Columns after feature engineering: ['NORAD_CAT_ID', 'EPOCH_UNIX', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE', 'is_debris']
Data split for Random Forest complete.
LSTM features available: ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
Data split and sequence creation for LSTM co

In [ ]:
# --- Phase 2: Random Forest and LSTM Models ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# --- Random Forest Regressor ---
print("\nTraining and evaluating the Random Forest Regressor...")
model_rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_rf.fit(X_train_rf, y_train_rf)
y_pred_rf = model_rf.predict(X_test_rf)
mse_rf = mean_squared_error(y_test_rf, y_pred_rf)
r_squared_rf = r2_score(y_test_rf, y_pred_rf)
mae_rf = mean_absolute_error(y_test_rf, y_pred_rf)
print(f'Random Forest R-squared: {r_squared_rf:.4f}')

# --- LSTM Model ---
print("\nTraining and evaluating the LSTM model...")
def create_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        X.append(data[i:(i + n_steps), :])
        y.append(data[i + n_steps, :])
    return np.array(X), np.array(y)

features_lstm = ['MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'ORBITAL_PERIOD', 'SEMI_MAJOR_AXIS', 'ORBITAL_VELOCITY', 'PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE']
scaler_lstm = MinMaxScaler()
train_scaled_lstm = scaler_lstm.fit_transform(train_df[features_lstm])
test_scaled_lstm = scaler_lstm.transform(test_df[features_lstm])
n_steps = 10
X_train_lstm, y_train_lstm = create_sequences(train_scaled_lstm, n_steps)
X_test_lstm, y_test_lstm = create_sequences(test_scaled_lstm, n_steps)
n_features = len(features_lstm)
n_outputs = len(features_lstm)
model_lstm = Sequential()
model_lstm.add(LSTM(50, activation='relu', input_shape=(n_steps, n_features)))
model_lstm.add(Dense(n_outputs))
model_lstm.compile(optimizer='adam', loss='mse')
model_lstm.fit(X_train_lstm, y_train_lstm, epochs=20, verbose=1)
y_pred_lstm = model_lstm.predict(X_test_lstm)
mse_lstm = mean_squared_error(y_test_lstm, y_pred_lstm)

print(f"\nLSTM Mean Squared Error: {mse_lstm:.4f}")

In [ ]:
# --- Phase 3: 3D Visualization ---
import plotly.graph_objects as go
import numpy as np
import pandas as pd

print("\nGenerating 3D Visualization (simplified)...")

# Read the main dataframe - assuming the updated df from the previous cell is available
# If running this cell independently, you might need to reload the data
# try:
#     df = pd.read_csv('space_debris_ml_ready.csv') # <--- Update this if your new file has a different name
# except FileNotFoundError:
#     print("Error: 'space_debris_ml_ready.csv' not found. Please upload it.")
#     # exit() # Uncomment this line to stop execution if the file is not found
#     df = None # Set df to None if file not found to prevent further errors

if df is not None and 'SEMI_MAJOR_AXIS' in df.columns and 'is_debris' in df.columns:
    # Prepare for plotting
    # Ensure 'is_debris' is calculated if not already present
    if 'is_debris' not in df.columns and 'BSTAR' in df.columns:
         df['is_debris'] = (df['BSTAR'] > 0.0001).astype(int)
         print("Engineered 'is_debris' for visualization.")
    elif 'is_debris' not in df.columns:
         print("Warning: 'is_debris' column not available for visualization.")


    # Use engineered features for a simplified 3D representation
    # We can use SEMI_MAJOR_AXIS to represent a characteristic orbital radius
    # For a simplified visualization, let's create points on a sphere with radius equal to SEMI_MAJOR_AXIS
    # This is a placeholder and does not represent the true orbital path

    # Calculate a simplified position based on semi-major axis and a hypothetical position
    # We will use random angles for this simplified visualization to show something in 3D
    np.random.seed(42) # for reproducibility
    num_points_to_plot = 1000

    # Select relevant columns and take the head
    # Ensure columns exist before selecting
    cols_to_plot = ['SEMI_MAJOR_AXIS', 'is_debris']
    available_cols_to_plot = [col for col in cols_to_plot if col in df.columns]

    if len(available_cols_to_plot) == len(cols_to_plot):
        df_plot_subset = df[available_cols_to_plot].head(num_points_to_plot).copy()

        r = df_plot_subset['SEMI_MAJOR_AXIS']
        theta = np.random.rand(len(r)) * 2 * np.pi # Random angle
        phi = np.random.rand(len(r)) * np.pi # Random angle

        x = r * np.sin(phi) * np.cos(theta)
        y = r * np.sin(phi) * np.sin(theta)
        z = r * np.cos(phi)

        df_plot = pd.DataFrame({'x': x, 'y': y, 'z': z, 'is_debris': df_plot_subset['is_debris']})

        # Create the 3D plot
        fig = go.Figure(data=[go.Scatter3d(x=df_plot['x'], y=df_plot['y'], z=df_plot['z'], mode='markers',
                                        marker=dict(size=2, color=df_plot['is_debris'], colorscale='Viridis', opacity=0.8))])
        fig.update_layout(title='Simplified 3D Visualization of Orbits (based on Semi-Major Axis)',
                        scene=dict(xaxis_title='X (km)', yaxis_title='Y (km)', zaxis_title='Z (km)'),
                        margin=dict(l=0, r=0, b=0, t=40))
        fig.show()
    else:
        missing = [col for col in cols_to_plot if col not in df.columns]
        print(f"Error: Cannot generate 3D plot. Missing required columns: {missing}")

else:
    print("Error: Dataframe not loaded or required columns ('SEMI_MAJOR_AXIS', 'is_debris') not available for 3D plot.")

In [ ]:
import pandas as pd

try:
    tle_df_combined = pd.read_csv('/content/space_debris_combined.csv')
    print("Columns in space_debris_combined.csv:", tle_df_combined.columns.tolist())
except FileNotFoundError:
    print("Error: 'space_debris_combined.csv' not found. Please ensure it's uploaded.")
    tle_df_combined = None

In [ ]:
print("NORAD_CAT_IDs in df_subset:")
print(df_subset['NORAD_CAT_ID'].unique())

print("\nNORAD_CAT_IDs in tle_data:")
print(list(tle_data.keys()))

In [1]:
import pandas as pd

try:
    tle_df_combined = pd.read_csv('/content/space_debris_combined.csv')
    print("Columns in space_debris_combined.csv:", tle_df_combined.columns.tolist())
except FileNotFoundError:
    print("Error: 'space_debris_combined.csv' not found. Please ensure it's uploaded.")
    tle_df_combined = None

Columns in space_debris_combined.csv: ['NORAD_CAT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'APOAPSIS', 'PERIAPSIS', 'BSTAR', 'COUNTRY_CODE', 'RCS_SIZE', 'NAME', 'LINE1', 'LINE2']


In [ ]:
print("NORAD_CAT_IDs in df_subset:")
print(df_subset['NORAD_CAT_ID'].unique())

print("\nNORAD_CAT_IDs in tle_data:")
print(list(tle_data.keys()))

# Task
Classify space objects as high-risk based on their orbital parameters using the pandas DataFrame `df` from the data loading step. Engineer a binary target column `is_high_risk` where an object is high-risk (1) if its `PERIGEE_ALTITUDE` is less than 500 km, and low-risk (0) otherwise. Define the feature set `X` using the columns ['PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE', 'INCLINATION', 'ECCENTRICITY', 'ORBITAL_VELOCITY'] and the target `y` as the `is_high_risk` column. Split `X` and `y` into training and testing sets with a test size of 30% and a random_state of 42. Initialize and train a `RandomForestClassifier` on the training data. Generate and print a `classification_report` and a `confusion_matrix` to evaluate the classifier's performance on the test data.

## Engineer risk feature

### Subtask:
Create the `is_high_risk` binary column based on `PERIGEE_ALTITUDE`.


**Reasoning**:
Create the binary 'is_high_risk' column based on the 'PERIGEE_ALTITUDE' threshold.



In [7]:
df['is_high_risk'] = (df['PERIGEE_ALTITUDE'] < 500).astype(int)
print("Engineered 'is_high_risk' column.")
display(df[['PERIGEE_ALTITUDE', 'is_high_risk']].head())

Engineered 'is_high_risk' column.


,PERIGEE_ALTITUDE,is_high_risk
0,-4856.464,1
1,-5549.134,1
2,-5647.848,1
3,-5560.429,1
4,-5662.528,1


## Prepare data

### Subtask:
Define the feature set `X` and the target `y` using the specified columns and the new `is_high_risk` column.


**Reasoning**:
Define the features and target for the classification task and check for missing columns before creating the feature and target dataframes.



In [8]:
features = ['PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE', 'INCLINATION', 'ECCENTRICITY', 'ORBITAL_VELOCITY']

# Check if all columns in the features list exist in the DataFrame df
missing_features = [col for col in features if col not in df.columns]

if missing_features:
    print(f"Error: The following feature columns are missing from the DataFrame: {missing_features}")
else:
    # Create the feature DataFrame X and target Series y
    X = df[features]
    y = df['is_high_risk']

    # Print the shapes to verify
    print("Shape of X:", X.shape)
    print("Shape of y:", y.shape)

Shape of X: (9669, 5)
Shape of y: (9669,)


## Split data

### Subtask:
Split the data into training and testing sets.


**Reasoning**:
Split the feature and target dataframes into training and testing sets using train_test_split with a test size of 0.3 and random state of 42.



In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (6768, 5)
Shape of X_test: (2901, 5)
Shape of y_train: (6768,)
Shape of y_test: (2901,)


## Train classifier

### Subtask:
Initialize and train a `RandomForestClassifier`.


**Reasoning**:
Initialize and train a RandomForestClassifier using the previously split training data.



In [10]:
from sklearn.ensemble import RandomForestClassifier

model_rf_classifier = RandomForestClassifier(random_state=42)
model_rf_classifier.fit(X_train, y_train)
print("RandomForestClassifier model trained.")

RandomForestClassifier model trained.


## Evaluate classifier

### Subtask:
Generate and print the classification report and confusion matrix.


**Reasoning**:
Generate predictions on the test set and then compute and print the classification report and confusion matrix to evaluate the classifier's performance.



In [11]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model_rf_classifier.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       100
           1       1.00      1.00      1.00      2801

    accuracy                           1.00      2901
   macro avg       0.99      0.99      0.99      2901
weighted avg       1.00      1.00      1.00      2901


Confusion Matrix:
[[  99    1]
 [   1 2800]]


## Summary:

### Data Analysis Key Findings

*   A binary target variable `is_high_risk` was successfully engineered based on the `PERIGEE_ALTITUDE` column, where objects with a `PERIGEE_ALTITUDE` less than 500 km were classified as high-risk (1) and others as low-risk (0).
*   The feature set `X` was successfully defined using the columns `['PERIGEE_ALTITUDE', 'APOAPSIS_ALTITUDE', 'INCLINATION', 'ECCENTRICITY', 'ORBITAL_VELOCITY']`, and the target `y` was set to the `is_high_risk` column. `X` has a shape of (9669, 5) and `y` has a shape of (9669,).
*   The data was split into training and testing sets with a 70/30 ratio. `X_train` has shape (6768, 5), `X_test` has shape (2901, 5), `y_train` has shape (6768,), and `y_test` has shape (2901,).
*   A `RandomForestClassifier` was initialized and trained on the training data, demonstrating excellent performance on the test set.
*   The classification report showed high precision, recall, and f1-scores for both high-risk and low-risk classes, indicating accurate classification.
*   The confusion matrix confirmed the model's accuracy with a low number of misclassifications.

### Insights or Next Steps

*   The trained Random Forest model effectively classifies space objects into high-risk and low-risk categories based on the provided orbital parameters and the defined risk criterion.
*   Further analysis could involve exploring feature importance from the trained model to understand which orbital parameters contribute most significantly to the risk classification.


# Task
Create an interactive 3D visualization of space object orbits using plotly.graph_objects. Generate a sample pandas DataFrame `orbits_df` with columns ['object_id', 'timestamp', 'x_km', 'y_km', 'z_km'] simulating paths for 3 objects. Plot these paths in 3D, color-coding each object's path uniquely. Include a central sphere representing the Earth (radius 6371 km, light blue). Configure hover data to show object_id and coordinates. Set the title to "Interactive 3D Orbit Visualization", use the 'plotly_dark' template, and ensure equal axis aspect ratios. Display the plot.

## Create sample data

### Subtask:
Generate a sample pandas DataFrame `orbits_df` with simulated orbital paths for 3 distinct objects.


**Reasoning**:
Generate a sample pandas DataFrame `orbits_df` with simulated orbital paths for 3 distinct objects by creating temporary dataframes for each object and then concatenating them.



In [17]:
import pandas as pd
import numpy as np

# List of object IDs
object_ids = ['object_1', 'object_2', 'object_3']

# Generate a sequence of timestamps
num_timepoints = 100
timestamps = pd.date_range(start='2023-01-01', periods=num_timepoints, freq='H')

# Simulate orbital paths for each object
all_orbits_data = []

for obj_id in object_ids:
    # Simulate orbital parameters
    amplitude_x = np.random.uniform(7000, 10000) # Varying semi-major axis
    amplitude_y = np.random.uniform(7000, 10000)
    amplitude_z = np.random.uniform(5000, 8000)
    frequency_x = np.random.uniform(0.1, 0.5) # Varying orbital period
    frequency_y = np.random.uniform(0.1, 0.5)
    frequency_z = np.random.uniform(0.1, 0.5)
    noise_level = np.random.uniform(50, 200) # Add some noise

    # Generate coordinates
    t = np.arange(num_timepoints)
    x_km = amplitude_x * np.cos(2 * np.pi * frequency_x * t / num_timepoints) + np.random.normal(0, noise_level, num_timepoints)
    y_km = amplitude_y * np.sin(2 * np.pi * frequency_y * t / num_timepoints) + np.random.normal(0, noise_level, num_timepoints)
    z_km = amplitude_z * np.sin(2 * np.pi * frequency_z * t / num_timepoints + np.pi/4) + np.random.normal(0, noise_level, num_timepoints) # Add phase shift for z

    # Create temporary DataFrame for the current object
    temp_df = pd.DataFrame({
        'object_id': obj_id,
        'timestamp': timestamps,
        'x_km': x_km,
        'y_km': y_km,
        'z_km': z_km
    })

    all_orbits_data.append(temp_df)

# Concatenate all temporary DataFrames
orbits_df = pd.concat(all_orbits_data, ignore_index=True)

# Print the head of the DataFrame
display(orbits_df.head())

/tmp/ipython-input-2233938007.py:9: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,object_id,timestamp,x_km,y_km,z_km
0,object_1,2023-01-01 00:00:00,7723.323412,-139.662426,5245.382011
1,object_1,2023-01-01 01:00:00,7782.856738,71.560761,5513.877162
2,object_1,2023-01-01 02:00:00,7794.602126,206.912184,5900.918518
3,object_1,2023-01-01 03:00:00,7646.744083,273.206756,5815.832462
4,object_1,2023-01-01 04:00:00,7568.237923,616.595335,6086.863988


## Initialize plotly figure

### Subtask:
Create a base Plotly figure object.


**Reasoning**:
Create an empty Plotly figure object to start building the 3D visualization.



In [18]:
import plotly.graph_objects as go

# Create an empty figure
fig = go.Figure()

print("Plotly figure object created.")

Plotly figure object created.


**Reasoning**:
Add a sphere to the figure to represent the Earth.



In [19]:
# Add a sphere representing Earth
R_EARTH = 6371.0 # Earth's radius in km
fig.add_trace(go.Sphere(
    r=R_EARTH,
    surface_color='lightblue',
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure.")

AttributeError: module 'plotly.graph_objects' has no attribute 'Sphere'

**Reasoning**:
The `go.Sphere` object is not available in `plotly.graph_objects`. Use `go.Surface` to create a sphere by generating coordinates for a sphere manually.



In [20]:
# Add a sphere representing Earth using go.Surface
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure using go.Surface.")

Earth sphere added to the figure using go.Surface.


**Reasoning**:
Iterate through the unique object IDs in the `orbits_df` DataFrame and add a scatter3d trace for each object's trajectory to the figure, using a distinct color for each object.



In [21]:
# Add traces for each object's orbit
unique_object_ids = orbits_df['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = orbits_df[orbits_df['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Orbital traces added to the figure.")

NameError: name 'plotly' is not defined

**Reasoning**:
The error indicates that the `plotly` module was not directly imported. Import `plotly.colors` to access the color palettes.



In [22]:
import plotly.colors

# Add traces for each object's orbit
unique_object_ids = orbits_df['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = orbits_df[orbits_df['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Orbital traces added to the figure.")

Orbital traces added to the figure.


**Reasoning**:
Update the figure layout with the specified title, template, and axis settings, and then display the plot.



In [23]:
# Update layout for better visualization
fig.update_layout(
    title='Interactive 3D Orbit Visualization',
    template='plotly_dark', # Use dark theme
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data' # Ensure equal aspect ratios
    ),
    margin=dict(l=0, r=0, b=0, t=40) # Adjust margins
)

# Display the plot
fig.show()

print("Plot layout updated and figure displayed.")

Plot layout updated and figure displayed.


## Summary:

### Data Analysis Key Findings

*   A sample pandas DataFrame `orbits_df` was successfully generated, containing simulated 3D orbital data for 3 distinct objects over 100 timepoints.
*   An interactive 3D Plotly figure was created, including a central sphere representing Earth with a radius of 6371 km.
*   Separate 3D line traces were added to the plot for each simulated object's orbit, color-coded using Plotly's qualitative color palette.
*   Hover information was configured for each orbital trace to display the object ID, timestamp, and coordinates (x, y, z) when the user hovers over the lines.
*   The plot layout was updated to include the title "Interactive 3D Orbit Visualization", apply the 'plotly\_dark' template, and ensure equal aspect ratios for the x, y, and z axes.

### Insights or Next Steps

*   The current simulation uses simplified trigonometric paths; future steps could involve implementing more realistic orbital mechanics (e.g., Keplerian orbits) for the simulation data.
*   Consider adding interactive elements like a time slider to animate the orbits or allow filtering by object ID for more detailed analysis.


# Task
Write a Python script that defines a function `generate_keplerian_orbit` to calculate ECI coordinates from Keplerian elements, implements the necessary orbital mechanics calculations within the function, generates orbital data for a LEO and a GEO object using this function, combines the data into a pandas DataFrame named `realistic_orbits_df` with columns ['object_id', 'timestamp', 'x_km', 'y_km', 'z_km'], and then visualizes this data using the provided Plotly 3D scatter plot code, including a sphere for Earth.

## Define keplerian orbit function

### Subtask:
Write a Python function `generate_keplerian_orbit` that accepts Keplerian elements and calculates ECI coordinates over one orbit.


**Reasoning**:
Define the `generate_keplerian_orbit` function with placeholder comments for Keplerian elements and return placeholder values for ECI coordinates.



In [31]:
import numpy as np

def generate_keplerian_orbit(semi_major_axis, eccentricity, inclination_deg, raan_deg, arg_of_perigee_deg, true_anomaly_start_deg, num_points=100):
    """
    Calculates Earth-Centered Inertial (ECI) coordinates for one orbit
    using Keplerian elements.

    Args:
        semi_major_axis (float): Semi-major axis in km.
        eccentricity (float): Eccentricity (unitless).
        inclination_deg (float): Inclination in degrees.
        raan_deg (float): Right Ascension of the Ascending Node in degrees.
        arg_of_perigee_deg (float): Argument of Perigee in degrees.
        true_anomaly_start_deg (float): Starting True Anomaly in degrees.
        num_points (int): Number of points to generate for the orbit.

    Returns:
        tuple: A tuple containing three numpy arrays (x_km, y_km, z_km)
               representing the ECI coordinates in kilometers over one orbit.
               Returns None if input is invalid.
    """
    # Placeholder for actual orbital mechanics calculations
    # These calculations will involve converting Keplerian elements to
    # position vectors over a range of time steps (or true anomaly).
    # For a full orbit, true anomaly will range from 0 to 360 degrees.

    # Basic validation (simplified)
    if semi_major_axis <= 0 or eccentricity < 0:
        print("Invalid Keplerian elements.")
        return None

    # Placeholder for calculating time steps or true anomaly range
    # For a full orbit, we can iterate through true anomaly from 0 to 360 degrees
    true_anomaly_range = np.linspace(np.deg2rad(true_anomaly_start_deg), np.deg2rad(true_anomaly_start_deg + 360), num_points)

    # Placeholder for ECI coordinate calculations based on orbital mechanics
    # These would involve rotation matrices based on inclination, RAAN, and arg_of_perigee,
    # and calculating radial distance based on true anomaly and eccentricity.
    # For now, return simulated circular orbit data as a placeholder
    r = semi_major_axis * (1 - eccentricity**2) / (1 + eccentricity * np.cos(true_anomaly_range))

    # Simplified ECI calculation ignoring inclination, RAAN, arg_of_perigee for now
    # This will be updated in the next subtask
    x_km = r * np.cos(true_anomaly_range)
    y_km = r * np.sin(true_anomaly_range)
    z_km = np.zeros_like(true_anomaly_range) # Placeholder: assuming orbit in xy-plane initially


    # Placeholder for converting to ECI using full orbital mechanics

    # For now, return the simplified data
    return x_km, y_km, z_km

print("Defined generate_keplerian_orbit function.")

Defined generate_keplerian_orbit function.


**Reasoning**:
Implement the necessary orbital mechanics calculations within the `generate_keplerian_orbit` function to correctly calculate ECI coordinates from Keplerian elements.



In [32]:
def generate_keplerian_orbit(semi_major_axis, eccentricity, inclination_deg, raan_deg, arg_of_perigee_deg, true_anomaly_start_deg, num_points=100):
    """
    Calculates Earth-Centered Inertial (ECI) coordinates for one orbit
    using Keplerian elements.

    Args:
        semi_major_axis (float): Semi-major axis in km.
        eccentricity (float): Eccentricity (unitless).
        inclination_deg (float): Inclination in degrees.
        raan_deg (float): Right Ascension of the Ascending Node in degrees.
        arg_of_perigee_deg (float): Argument of Perigee in degrees.
        true_anomaly_start_deg (float): Starting True Anomaly in degrees.
        num_points (int): Number of points to generate for the orbit.

    Returns:
        tuple: A tuple containing three numpy arrays (x_km, y_km, z_km)
               representing the ECI coordinates in kilometers over one orbit.
               Returns None if input is invalid.
    """
    # Basic validation
    if semi_major_axis <= 0 or eccentricity < 0:
        print("Invalid Keplerian elements.")
        return None

    # Convert angles to radians
    inclination_rad = np.deg2rad(inclination_deg)
    raan_rad = np.deg2rad(raan_deg)
    arg_of_perigee_rad = np.deg2rad(arg_of_perigee_deg)
    true_anomaly_start_rad = np.deg2rad(true_anomaly_start_deg)


    # Generate a range of true anomaly values for one orbit
    true_anomaly_range_rad = np.linspace(true_anomaly_start_rad, true_anomaly_start_rad + 2 * np.pi, num_points)

    # Calculate radial distance (r) for each true anomaly
    r = semi_major_axis * (1 - eccentricity**2) / (1 + eccentricity * np.cos(true_anomaly_range_rad))

    # Calculate position in the orbital plane (perifocal frame)
    x_perifocal = r * np.cos(true_anomaly_range_rad)
    y_perifocal = r * np.sin(true_anomaly_range_rad)
    z_perifocal = np.zeros_like(true_anomaly_range_rad)

    # Convert from perifocal frame to ECI frame using rotation matrices
    # Rotation 1: Rotate by argument of perigee around Z-axis
    # Rotation 2: Rotate by inclination around X-axis
    # Rotation 3: Rotate by RAAN around Z-axis

    # Combined rotation matrix (Z-X-Z Euler angles)
    cos_raan = np.cos(raan_rad)
    sin_raan = np.sin(raan_rad)
    cos_inc = np.cos(inclination_rad)
    sin_inc = np.sin(inclination_rad)
    cos_argp = np.cos(arg_of_perigee_rad)
    sin_argp = np.sin(arg_of_perigee_rad)

    # R_z(RAAN) * R_x(inc) * R_z(arg_of_perigee)
    # First column of the combined rotation matrix
    R11 = cos_raan * cos_argp - sin_raan * sin_argp * cos_inc
    R21 = sin_raan * cos_argp + cos_raan * sin_argp * cos_inc
    R31 = sin_argp * sin_inc

    # Second column of the combined rotation matrix
    R12 = -cos_raan * sin_argp - sin_raan * cos_argp * cos_inc
    R22 = -sin_raan * sin_argp + cos_raan * cos_argp * cos_inc
    R32 = cos_argp * sin_inc

    # Third column of the combined rotation matrix
    R13 = sin_raan * sin_inc
    R23 = -cos_raan * sin_inc
    R33 = cos_inc

    # Apply the rotation matrix to the perifocal coordinates
    x_eci = R11 * x_perifocal + R12 * y_perifocal + R13 * z_perifocal
    y_eci = R21 * x_perifocal + R22 * y_perifocal + R23 * z_perifocal
    z_eci = R31 * x_perifocal + R32 * y_perifocal + R33 * z_perifocal

    return x_eci, y_eci, z_eci

print("Updated generate_keplerian_orbit function with orbital mechanics.")

Updated generate_keplerian_orbit function with orbital mechanics.


**Reasoning**:
Generate orbital data for a LEO and a GEO object using the `generate_keplerian_orbit` function and combine the data into a pandas DataFrame.



In [33]:
import pandas as pd
import numpy as np

# --- Generate Orbital Data ---

# Define Keplerian elements for a sample LEO object
# Approximate values for ISS or similar LEO
leo_semi_major_axis = 6371.0 + 400.0  # Earth radius + ~400 km altitude
leo_eccentricity = 0.0005
leo_inclination_deg = 51.6
leo_raan_deg = 250.0
leo_arg_of_perigee_deg = 90.0
leo_true_anomaly_start_deg = 0.0

# Define Keplerian elements for a sample GEO object
# Approximate values for GEO
geo_semi_major_axis = 42164.0 # ~35786 km altitude + Earth radius
geo_eccentricity = 0.0001
geo_inclination_deg = 0.01 # Ideally close to 0 for GEO
geo_raan_deg = 0.0
geo_arg_of_perigee_deg = 0.0
geo_true_anomaly_start_deg = 0.0

# Generate data for LEO object
x_leo, y_leo, z_leo = generate_keplerian_orbit(
    semi_major_axis=leo_semi_major_axis,
    eccentricity=leo_eccentricity,
    inclination_deg=leo_inclination_deg,
    raan_deg=leo_raan_deg,
    arg_of_perigee_deg=leo_arg_of_perigee_deg,
    true_anomaly_start_deg=leo_true_anomaly_start_deg,
    num_points=200 # More points for smoother visualization
)

# Generate data for GEO object
x_geo, y_geo, z_geo = generate_keplerian_orbit(
    semi_major_axis=geo_semi_major_axis,
    eccentricity=geo_eccentricity,
    inclination_deg=geo_inclination_deg,
    raan_deg=geo_raan_deg,
    arg_of_perigee_deg=geo_arg_of_perigee_deg,
    true_anomaly_start_deg=geo_true_anomaly_start_deg,
    num_points=200 # More points for smoother visualization
)

# Combine data into a single DataFrame
leo_df = pd.DataFrame({
    'object_id': 'LEO_Object',
    'timestamp': pd.to_datetime(np.linspace(0, 24*3600, len(x_leo)), unit='s'), # Simulate timestamps over a period
    'x_km': x_leo,
    'y_km': y_leo,
    'z_km': z_leo
})

geo_df = pd.DataFrame({
    'object_id': 'GEO_Object',
    'timestamp': pd.to_datetime(np.linspace(0, 24*3600*365, len(x_geo)), unit='s'), # Simulate timestamps over a longer period for GEO relative motion
    'x_km': x_geo,
    'y_km': y_geo,
    'z_km': z_geo
})

realistic_orbits_df = pd.concat([leo_df, geo_df], ignore_index=True)

print("Generated orbital data for LEO and GEO objects.")
display(realistic_orbits_df.head())

Generated orbital data for LEO and GEO objects.


,object_id,timestamp,x_km,y_km,z_km
0,LEO_Object,1970-01-01 00:00:00.000000000,3950.175274,-1437.746220,5303.735205
1,LEO_Object,1970-01-01 00:07:14.170854271,4021.277940,-1236.270461,5301.093086
2,LEO_Object,1970-01-01 00:14:28.341708543,4088.374179,-1033.562774,5293.169356
3,LEO_Object,1970-01-01 00:21:42.512562814,4151.397199,-829.824918,5279.971888
4,LEO_Object,1970-01-01 00:28:56.683417085,4210.284255,-625.259677,5261.513799


**Reasoning**:
Initialize a Plotly figure object to prepare for adding 3D traces.



In [34]:
import plotly.graph_objects as go

# Create an empty figure
fig = go.Figure()

print("Plotly figure object created.")

Plotly figure object created.


**Reasoning**:
Add a sphere to the figure to represent the Earth using `go.Surface` and manually generated coordinates, as `go.Sphere` is not available in `plotly.graph_objects`.



In [35]:
# Add a sphere representing Earth using go.Surface
R_EARTH = 6371.0 # Earth's radius in km

# Generate coordinates for a sphere
theta = np.linspace(0, 2*np.pi, 100)
phi = np.linspace(0, np.pi, 100)
theta, phi = np.meshgrid(theta, phi)

x_earth = R_EARTH * np.sin(phi) * np.cos(theta)
y_earth = R_EARTH * np.sin(phi) * np.sin(theta)
z_earth = R_EARTH * np.cos(phi)

fig.add_trace(go.Surface(
    x=x_earth,
    y=y_earth,
    z=z_earth,
    colorscale=[[0, 'lightblue'], [1, 'lightblue']], # Solid light blue color
    showscale=False, # Hide color scale
    opacity=0.8,
    name='Earth'
))

print("Earth sphere added to the figure using go.Surface.")

Earth sphere added to the figure using go.Surface.


**Reasoning**:
Iterate through the unique object IDs in the `realistic_orbits_df` DataFrame and add a scatter3d trace for each object's trajectory to the figure, using a distinct color for each object and configuring hover information.



In [36]:
import plotly.colors

# Add traces for each object's orbit
unique_object_ids = realistic_orbits_df['object_id'].unique()

# Define a color palette (using Plotly's default categorical colors)
colors = plotly.colors.qualitative.Plotly

for i, obj_id in enumerate(unique_object_ids):
    obj_df = realistic_orbits_df[realistic_orbits_df['object_id'] == obj_id]
    fig.add_trace(go.Scatter3d(
        x=obj_df['x_km'],
        y=obj_df['y_km'],
        z=obj_df['z_km'],
        mode='lines',
        name=obj_id,
        line=dict(color=colors[i % len(colors)], width=2),
        hoverinfo='text',
        text=[f'Object ID: {row["object_id"]}<br>Timestamp: {row["timestamp"]}<br>X: {row["x_km"]:.2f}<br>Y: {row["y_km"]:.2f}<br>Z: {row["z_km"]:.2f}'
              for index, row in obj_df.iterrows()]
    ))

print("Orbital traces added to the figure.")

Orbital traces added to the figure.


**Reasoning**:
Update the figure layout with the specified title, template, and axis settings, and then display the plot to complete the visualization subtask.



In [37]:
# Update layout for better visualization
fig.update_layout(
    title='Interactive 3D Orbit Visualization',
    template='plotly_dark', # Use dark theme
    scene=dict(
        xaxis=dict(title='X (km)'),
        yaxis=dict(title='Y (km)'),
        zaxis=dict(title='Z (km)'),
        aspectmode='data' # Ensure equal aspect ratios
    ),
    margin=dict(l=0, r=0, b=0, t=40) # Adjust margins
)

# Display the plot
fig.show()

print("Plot layout updated and figure displayed.")

Plot layout updated and figure displayed.


## Summary:

### Data Analysis Key Findings

*   The `generate_keplerian_orbit` function was successfully implemented to calculate ECI coordinates from Keplerian elements using orbital mechanics principles.
*   Orbital data for a sample Low Earth Orbit (LEO) object with a semi-major axis of approximately 6771 km and for a sample Geostationary Orbit (GEO) object with a semi-major axis of approximately 42164 km were generated.
*   The generated orbital data for both LEO and GEO objects were successfully combined into a single pandas DataFrame named `realistic_orbits_df`.
*   An interactive 3D visualization was created using Plotly, successfully plotting the calculated orbits of both the LEO and GEO objects and including a sphere representing Earth.

### Insights or Next Steps

*   The current simulation assumes a single orbit under ideal Keplerian conditions. A valuable next step would be to extend the `generate_keplerian_orbit` function or add subsequent processing to simulate multiple orbits and incorporate orbital perturbations (e.g., atmospheric drag for LEO, solar radiation pressure, gravitational effects from other bodies) for a more realistic long-term trajectory.
*   Enhance the visualization by adding features like animating the orbits over time, displaying the current position of the satellites along the orbit, or adding labels for perigee and apogee points.
